In [1]:
%%writefile check_semantic.py
import os
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
import polars as pl
from scipy.special import softmax

SYS_PRMPT = """
You are an experienced, fair, unbiased moderator. 
Classify whether the comment violates the supplied moderation rule.
- True: the comment breaks the specified rule
- False: the comment is considered safe in relation to the specified rule
Respond only using True or False.
""".strip()

# Updated template to use semantic examples
USR_PRMPT_TMPLT = """
[RULE]: {}
[True EXAMPLE]: {}
[False EXAMPLE]: {}
[True EXAMPLE]: {}
[False EXAMPLE]: {}
[TEST CASE COMMENT]: {}
""".strip()

CHOICES = ['True', 'False']

def chat_formatting(df, tokenizer):
  prompts = []
  for user_content in df['user_content']:
    chat = [
      {'role': 'system', 'content': SYS_PRMPT},
      {'role': 'user', 'content': user_content.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
      chat, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    prompts.append(prompt)
  df = df.with_columns(pl.Series('prompt', prompts))
  return df


if __name__ == '__main__':
  print('importing vllm...')
  import vllm
  from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
  print('loading vllm...')

  llm = vllm.LLM(
    '/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1',
    quantization='awq',
    task='generate',
    tensor_parallel_size=2,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    enable_prefix_caching=True,
    dtype='half',
    enforce_eager=True,
    disable_log_stats=True,
    disable_custom_all_reduce=True,
  )
  print('building prompts...')
  
  # Load pre-computed semantic examples instead of raw test data
  # if IS_SUB:
    # For submission, use the semantic examples CSV
  path = 'test_with_semantic_examples_basicllm.csv'
  # else:
  #   # For training evaluation, we need to create semantic examples for train.csv
  #   # For now, use original path - could be enhanced later
  #   path = '/kaggle/input/jigsaw-agile-community-rules/train.csv'
  
  # if IS_SUB:
    # Use semantic examples for submission
  df = (
        pl.read_csv(path)
        .with_columns([pl.col(x).str.replace_all(r'\s+', ' ') for x in ["body", "^.*semantic.*example.*$"]])
        .with_columns([
            pl.col("semantic_positive_example_1").str.slice(0, 1000).alias("semantic_positive_example_1"),
            pl.col("semantic_positive_example_2").str.slice(0, 1000).alias("semantic_positive_example_2"),
            pl.col("semantic_negative_example_1").str.slice(0, 1000).alias("semantic_negative_example_1"),
            pl.col("semantic_negative_example_2").str.slice(0, 1000).alias("semantic_negative_example_2")
        ])
        .with_columns(pl.format(USR_PRMPT_TMPLT,
                               "rule",
                               "semantic_positive_example_1",
                               "semantic_negative_example_1",
                               "semantic_positive_example_2",
                               "semantic_negative_example_2",
                               "body").alias('user_content'))
    )
  # else:
  #   # Fallback to original examples for train evaluation
  #   df = (
  #       pl.read_csv(path)
  #       .with_columns([pl.col(x).str.replace_all(r'\s+', ' ') for x in ["body", "^.*_example_.*$"]])
  #       .with_columns(pl.format(USR_PRMPT_TMPLT, 
  #                              "rule", 
  #                              "positive_example_1", 
  #                              "negative_example_1",
  #                              "positive_example_2", 
  #                              "negative_example_2", 
  #                              "body").alias('user_content'))
  #   )

  tokenizer = llm.get_tokenizer()
  df = chat_formatting(df, tokenizer)
  prompts = df['prompt'].to_list()
  mclp = MultipleChoiceLogitsProcessor(
    tokenizer,
    choices=CHOICES,
  )
  sampling_params_choice = vllm.SamplingParams(seed=1337, skip_special_tokens=True, max_tokens=1, logits_processors=[mclp], logprobs=len(mclp.choices),)
  outputs = llm.generate(prompts, sampling_params_choice, use_tqdm=True)
  logprobs = [
    {lp.decoded_token: lp.logprob for lp in list(lps)}
    for lps in [output.outputs[0].logprobs[0].values() for output in outputs]
  ]
  choices = [max(d, key=d.get) for d in logprobs]
  print('generation end')
  df = df.with_columns(pl.Series('logprobs', logprobs), pl.Series('type', choices))
  print(df.group_by('type').agg(pl.len()).sort(['type']))
  logprobs = df['logprobs'].to_numpy()
  probs = softmax(logprobs, axis=-1)
  sub = df.with_columns(pl.Series('rule_violation', probs[:,0].tolist()))
  sub.select('row_id', 'rule_violation').write_csv('submission.csv')
  if not IS_SUB:
    import pandas as pd
    from sklearn.metrics import roc_auc_score
    
    # Load CSVs
    submission = pd.read_csv("submission.csv")
    gt = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")[['row_id', 'rule_violation']]
    
    # Merge on 'row_id' to align predictions and ground truth
    merged = pd.merge(gt, submission, on="row_id", suffixes=('_gt', '_pred'))
    
    # Ensure proper columns
    y_true = merged['rule_violation_gt']
    y_score = merged['rule_violation_pred']
    
    # Compute AUC
    try:
        auc = roc_auc_score(y_true, y_score)
        print(f"Column-Averaged AUC: {auc:.6f}")
    except ValueError as e:
        print(f"Cannot compute AUC: {e}")

Writing check_semantic.py


In [2]:
%%writefile prepare_basicllm_semantic_data.py
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
import random
import numpy as np
import os
random.seed(42)
np.random.seed(42)
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))


# Constants
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
EMBEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
TOP_K_SEMANTIC = 100  # Reduced for speed
EMBEDDING_BATCH_SIZE = 128//4


def build_labeled_corpus(data_path):
    """Build comprehensive labeled corpus from all available data"""
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    corpus = []

    # Add train data
    for _, row in train_dataset.iterrows():
        corpus.append({
            "body": row["body"],
            "rule": row["rule"],
            "subreddit": row["subreddit"],
            "rule_violation": row["rule_violation"]
        })

    # Add positive examples from test data
    for _, row in test_dataset.iterrows():
        for i in [1, 2]:
            corpus.append({
                "body": row[f"positive_example_{i}"],
                "rule": row["rule"],
                "subreddit": row["subreddit"],
                "rule_violation": 1
            })

    # Add negative examples from test data
    for _, row in test_dataset.iterrows():
        for i in [1, 2]:
            corpus.append({
                "body": row[f"negative_example_{i}"],
                "rule": row["rule"],
                "subreddit": row["subreddit"],
                "rule_violation": 0
            })

    corpus_df = pd.DataFrame(corpus).drop_duplicates().reset_index(drop=True)
    corpus_df["corpus_id"] = corpus_df.index
    return corpus_df


def main():
    # Build corpus for semantic search
    print("Building labeled corpus...")
    corpus_df = build_labeled_corpus(DATA_PATH)
    
    # Load embedding model
    print("Loading embedding model...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_PATH, device="cuda")
    
    # Load test data
    if IS_SUB:
        test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    else:
        test_dataframe = pd.read_csv(f"{DATA_PATH}/train.csv")
    # Group test data by rule for efficient batch processing
    rule_groups = {}
    for _, row in test_dataframe.iterrows():
        rule = row["rule"]
        if rule not in rule_groups:
            rule_groups[rule] = []
        rule_groups[rule].append(row.to_dict())
    
    print(f"Processing {len(test_dataframe)} test examples across {len(rule_groups)} rules")
    
    all_test_data = []
    
    # Process each rule group with batch optimization
    for rule, test_rows in tqdm(rule_groups.items(), desc="Processing rule groups"):
        # Filter corpus by rule only (not subreddit for larger corpus)
        rule_corpus = corpus_df[corpus_df["rule"] == rule].reset_index(drop=True)
        
        if len(rule_corpus) == 0:
            # Fallback to entire corpus
            rule_corpus = corpus_df.reset_index(drop=True)
        
        # Pre-encode corpus once for this rule
        print(f"Encoding corpus for {rule[:50]}... ({len(rule_corpus)} examples)")
        corpus_embeddings = embedding_model.encode(
            rule_corpus["body"].tolist(),
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        
        # Extract all test bodies for batch encoding
        test_bodies = [row["body"] for row in test_rows]
        
        print(f"Encoding {len(test_bodies)} test comments...")
        test_embeddings = embedding_model.encode(
            test_bodies,
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        
        # Batch semantic search for all test examples in this rule
        search_results_batch = semantic_search(
            test_embeddings,
            corpus_embeddings,
            top_k=min(TOP_K_SEMANTIC, len(rule_corpus)),
            score_function=dot_score,
        )
        
        # Pre-filter corpus by violation type for faster lookup
        positive_corpus = rule_corpus[rule_corpus["rule_violation"] == 1].reset_index(drop=True)
        negative_corpus = rule_corpus[rule_corpus["rule_violation"] == 0].reset_index(drop=True)
        
        # Process results for each test example
        for i, test_row in enumerate(test_rows):
            search_results = search_results_batch[i]
            
            # Collect examples efficiently
            positive_examples = []
            negative_examples = []
            used_bodies = {test_row["body"]}  # Track used bodies to avoid duplicates
            
            for result in search_results:
                corpus_idx = result["corpus_id"]
                example_row = rule_corpus.iloc[corpus_idx]
                body = example_row["body"]
                
                # Skip if already used or exact match
                if body in used_bodies:
                    continue
                
                # Add to appropriate list
                if example_row["rule_violation"] == 1 and len(positive_examples) < 2:
                    positive_examples.append(body)
                    used_bodies.add(body)
                elif example_row["rule_violation"] == 0 and len(negative_examples) < 2:
                    negative_examples.append(body)
                    used_bodies.add(body)
                
                # Early exit when we have enough
                if len(positive_examples) >= 2 and len(negative_examples) >= 2:
                    break
            
            # Fast fallback using pre-filtered corpus
            while len(positive_examples) < 2 and len(positive_corpus) > 0:
                candidates = positive_corpus[~positive_corpus["body"].isin(used_bodies)]
                if len(candidates) > 0:
                    sample = candidates.sample(1)["body"].iloc[0]
                    positive_examples.append(sample)
                    used_bodies.add(sample)
                else:
                    positive_examples.append("No positive example found")
            
            while len(negative_examples) < 2 and len(negative_corpus) > 0:
                candidates = negative_corpus[~negative_corpus["body"].isin(used_bodies)]
                if len(candidates) > 0:
                    sample = candidates.sample(1)["body"].iloc[0]
                    negative_examples.append(sample)
                    used_bodies.add(sample)
                else:
                    negative_examples.append("No negative example found")
            
            # Ensure we have exactly 2 of each
            while len(positive_examples) < 2:
                positive_examples.append("No positive example found")
            while len(negative_examples) < 2:
                negative_examples.append("No negative example found")
            
            # Add semantic examples to test row
            test_result = test_row.copy()
            test_result["semantic_positive_example_1"] = positive_examples[0]
            test_result["semantic_positive_example_2"] = positive_examples[1]
            test_result["semantic_negative_example_1"] = negative_examples[0]
            test_result["semantic_negative_example_2"] = negative_examples[1]
            
            all_test_data.append(test_result)
        
        # Clear embeddings to save memory
        del corpus_embeddings, test_embeddings
        torch.cuda.empty_cache()
    
    # Save results
    result_df = pd.DataFrame(all_test_data)
    result_df.to_csv("test_with_semantic_examples_basicllm.csv", index=False)
    print(f"✅ Saved test_with_semantic_examples_basicllm.csv with {len(result_df)} examples")
    
    # Clear GPU memory
    del embedding_model
    torch.cuda.empty_cache()
    
    # Show a sample
    print("\nSample semantic examples:")
    sample = result_df.iloc[0]
    print(f"Rule: {sample['rule']}")
    print(f"Target: {sample['body'][:100]}...")
    print(f"Semantic Pos 1: {sample['semantic_positive_example_1'][:100]}...")
    print(f"Semantic Neg 1: {sample['semantic_negative_example_1'][:100]}...")


if __name__ == "__main__":
    main()

Writing prepare_basicllm_semantic_data.py


In [3]:
%%bash
WHEELHOUSE=/kaggle/input/llm-packages

uv pip uninstall --system 'tensorflow'
uv pip install -U --system --no-index --find-links=$WHEELHOUSE 'polars==1.31.0' 'vllm==0.10.0' 'numpy==1.26.4' 'logits-processor-zoo==0.2.1'
uv pip install -U --system --no-index --find-links=$WHEELHOUSE 'triton==3.2.0'

Using Python 3.11.13 environment at: /usr
Uninstalled 1 package in 5.16s
 - tensorflow==2.18.0
Using Python 3.11.13 environment at: /usr
Resolved 151 packages in 424ms
Prepared 106 packages in 37.84s
Uninstalled 74 packages in 6.09s
Installed 106 packages in 3.74s
 - accelerate==1.8.1
 + accelerate==1.10.1
 - aiohttp==3.12.13
 + aiohttp==3.12.15
 - aiosignal==1.3.2
 + aiosignal==1.4.0
 - anyio==4.9.0
 + anyio==4.11.0
 + astor==0.8.1
 + blake3==1.0.7
 - cachetools==5.5.2
 + cachetools==6.2.0
 + cbor2==5.7.0
 - certifi==2025.6.15
 + certifi==2025.8.3
 - cffi==1.17.1
 + cffi==2.0.0
 - charset-normalizer==3.4.2
 + charset-normalizer==3.4.3
 - click==8.2.1
 + click==8.3.0
 + compressed-tensors==0.10.2
 - cupy-cuda12x==13.4.1
 + cupy-cuda12x==13.6.0
 + depyf==0.19.0
 - dill==0.3.8
 + dill==0.4.0
 + diskcache==5.6.3
 - dnspython==2.7.0
 + dnspython==2.8.0
 - email-validator==2.2.0
 + email-validator==2.3.0
 - fastapi==0.115.13
 + fastapi==0.118.0
 + fastapi-cli==0.0.13
 + fastapi-cloud-cli==0

In [4]:
import os
IS_SUB=bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

In [5]:
if IS_SUB:
    !python prepare_basicllm_semantic_data.py
    ! VLLM_USE_V1=0 TORCH_CUDA_ARCH_LIST=7.5 python check_semantic.py
else:
    !touch submission.csv

In [6]:
! head -n 5 submission.csv